**Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Stretch: from Shiller's Excel file to a clean CSV

In the lecture, we loaded a ready-made CSV of Robert Shiller's data. This notebook shows how that CSV was made from the original Excel file on [shillerdata.com](https://shillerdata.com/).

Real data often arrives like this: titles above the table, multi-row headers, notes at the bottom, and dates stored as numbers. It's the kind of cleaning you'll need for your project.

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.
>
> Run a cell by clicking it and pressing **Shift + Enter**.

<br><br>

---

<br>

## 1. Packages

Reading old-style `.xls` files needs one extra package, `xlrd`:

In [ ]:
%pip install xlrd --quiet

import pandas as pd

<br><br>

---

<br>

## 2. Read the Excel file

A copy of the file is in the course repository. `pd.read_excel()` reads it, just like `read_csv()`. The spreadsheet has two sheets; the numbers are in the one called **Data**.

In [ ]:
xls_url = "https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week03/data/data_shiller.xls"

raw = pd.read_excel(xls_url, sheet_name="Data")
raw.head(10)

The first rows are titles and a header spread over several rows, so pandas has guessed the column names wrong.

The real column names (`Date`, `P`, `D`, ...) are on the 8th row of the spreadsheet. `skiprows=7` tells pandas to skip the 7 rows above them:

In [ ]:
raw = pd.read_excel(xls_url, sheet_name="Data", skiprows=7)
raw.head()

In [ ]:
raw.columns

Some names appear twice (pandas adds `.1` to the second one). Comparing with the spreadsheet:

| Column | Meaning |
|---|---|
| `Date` | Year.month |
| `P`, `D`, `E` | Price, dividends and earnings (S&P composite) |
| `CPI` | Consumer price index |
| `Rate GS10` | 10-year US government bond rate |
| `Price`, `Earnings` | *Real* (inflation-adjusted) price and earnings |
| `CAPE` | Cyclically adjusted P/E: real price ÷ 10-year average of real earnings |

<br><br>

---

<br>

## 3. Keep and rename the columns we want

In [ ]:
df = raw[['Date', 'P', 'D', 'E', 'CPI', 'Rate GS10', 'Price', 'Earnings', 'CAPE']]

df = df.rename(columns={'Price': 'Price_real', 'Earnings': 'Earnings_real'})
df.head()

<br><br>

---

<br>

## 4. Remove the notes at the bottom

The last row of the sheet is a note, not data:

In [ ]:
df.tail(3)

It has no `Date`, so we can drop rows where `Date` is missing:

In [ ]:
df = df.dropna(subset=['Date'])
df.tail(3)

<br><br>

---

<br>

## 5. Fix the dates

Excel stores the dates as **numbers**. Look at row 9, October 1871:

In [ ]:
df['Date'][9]

As a number, `1871.10` is the same as `1871.1`, so the zero is gone. (Tables like `df.head()` show 2 decimal places, which hides the problem.)

Read as a date, `1871.1` would mean January. We could fix this with some text manipulation, but there is a simpler way.

Every row is one month, in order, starting in January 1871. So we can **create** the dates, like using *Fill → Series* in Excel (Week 2). `pd.date_range()` makes a list of dates:
- `start=`: the first date
- `periods=`: how many dates (one per row, so `len(df)`)
- `freq='MS'`: one per **M**onth, on the first day (**S**tart) of the month

In [ ]:
df['Date_iso'] = pd.date_range(start='1871-01-01', periods=len(df), freq='MS')

df[['Date', 'Date_iso']][8:12]

**Check:** this only works if no months are missing. If they all line up, the last row should match too:

In [ ]:
df[['Date', 'Date_iso']].tail(3)

Finally, write the date back in Shiller's `yyyy.mm` style **as text**, so October is `1871.10`. (`.dt.strftime()` turns a date into text in any format, using the same `%Y` and `%m` codes as before.)

In [ ]:
df['Date'] = df['Date_iso'].dt.strftime('%Y.%m')

df = df.drop(columns=['Date_iso'])
df[8:12]

<br><br>

---

<br>

## 6. Save

This is the CSV we load in the lecture notebook (DS3_2).

In [ ]:
df.to_csv('data_shiller.csv', index=False)

In [ ]:
from google.colab import files
files.download('data_shiller.csv')

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1.** Shiller updates the data every month. Download the latest `.xls` from [shillerdata.com](https://shillerdata.com/), upload it to your own GitHub repository, and run this notebook on it.

**2.** Chart the `CAPE` column with Altair. How does it compare with the simple P/E from the seminar?

In [ ]:
# Your code here
